# Step-by-step demo of NEDAS using the vort2d model



## System setup (optional)

NEDAS is available from the PyPI platform. You can install with ```pip install NEDAS```.

You can also clone the `NEDAS` repository and install in editable mode ```cd NEDAS; pip install -e .``` for active code development.

In the docker image ``myying/nedas-tutorials`` the environment is ready, you can skip this part

In [ ]:
# In Google Colab, run this cell to install the environment.
# Only need to do this once, you may need to restart the kernel after this.
to_install = False

if to_install:
    # 1. Install the latest version of NEDAS on develop branch
    %cd ~
    %rm -rf NEDAS
    !git clone https://github.com/nansencenter/NEDAS.git
    %cd NEDAS
    %pip install .
    
    # 2. Install additional dependencies
    # numba provides JIT compilation of python function to speed it up during runtime
    %pip install numba
    # cmocean provides better colormaps for visualization
    %pip install cmocean
    # ipython widgets for interactive plots
    %pip install ipywidgets
    
    # 3. Clone the tutorial repo too
    %mkdir ~/work
    %cd ~/work
    %rm -rf NEDAS_tutorials
    !git clone https://github.com/myying/NEDAS_tutorials.git
    %cd ~/work/NEDAS_tutorials


## Initialization

A YAML file `vort2d/config.yml` contains all the settings for an experiment.

See [Configuration file](https://nedas.readthedocs.io/en/latest/config_file.html) documentation for more details.

In command line, you can run the experiment with `python -m NEDAS -c vort2d/config.yml`

Here we setup the `config` object in an interactive environment:

In [3]:
# utility modules
import numpy as np
from datetime import datetime, timezone
import matplotlib.pyplot as plt

# key NEDAS modules
from NEDAS.config import Config
from NEDAS.schemes import get_scheme

# some utility funcs for the vort2d case
from vort2d.utils import *
from vort2d.graphics import *

In [ ]:
# load configuration YAML file
# additional kwargs will overwrite the values in the file
config = Config(config_file="vort2d/config.yml", nproc=1, quiet=False)

# you can also change values like this
# config.nens = 1000
#config.debug = False
#config.io_mode = 'online'
#config.cycle_period = 6
# config.model_def['vort2d']['dt'] = 10
#config.model_def['vort2d']['loc_sprd'] = 10_000
#config.model_def['vort2d']['Vbg'] = 0

# config.obs_def[0]['hroi'] = 150_000
# config.obs_def[0]['nobs'] = 1
# config.dataset_def['vort2d']['network_type'] = 'targeted'
# config.dataset_def['vort2d']['obs_range'] = 50_000

# config.inflation_def['adaptive'] = False

# to list all configuration variables
#config.__dict__

In [ ]:
# Once you are happy with the configuration,
# you can initialize the main Scheme object
scheme = get_scheme(config)

## The verifying truth

We will run an Observing System Simulation Experiment (OSSE) for the vort2d model.

First step is to generate a "verifying truth", or a "nature run", which will be used both for generating synthetic observations and for verification of the results.

In [ ]:
%rm -rf vort2d/work/truth

# set time to the start of the experiment
scheme.c.time = config.time_start

# run the model from time_start to time_end and save results
# in offline IO mode, the truth states are saved under model.truth_dir (vort2d/work/truth/*nc files)
scheme.run_step('prepare_truth')

In [ ]:
truth_state = get_time_series(scheme.c, get_truth)

## Ensemble generation

Second,

In [ ]:
%rm -rf vort2d/work/init_ens

scheme.c.time = config.time_start

scheme.run_step('prepare_init_ensemble')

scheme.run_step('preprocess')

In [ ]:
init_ens = get_model_ens(scheme.c, config.time_start)

## Data assimilation


### Assimilating a single observation

In [ ]:
scheme.c.time = datetime(2001,1,1, tzinfo=timezone.utc)
scheme.run_step('preprocess')

%mkdir -p vort2d/work/cycle/200101010000/analysis

In [ ]:
from NEDAS.core import State, Obs

scheme.c.time = datetime(2001,1,1, tzinfo=timezone.utc)
scheme.c.progress.call_stack_max_level = None
scheme.c.progress.call_stack = []

scheme.c.progress.push('filter')

scheme.c.iter = 0
scheme.c.update_assim_tools()

scheme.c.state = State(scheme.c)
scheme.c.logger('Prepare prior state')(scheme.c.state.prepare_state)(scheme.c)

In [ ]:
scheme.c.obs = Obs(scheme.c)
scheme.c.logger('Prepare obs')(scheme.c.obs.prepare_obs)(scheme.c)
scheme.c.logger('Prepare obs from prior state')(scheme.c.obs.prepare_obs_from_state)(scheme.c, 'prior')

In [ ]:
from NEDAS.assim_tools.assimilators import get_assimilator

scheme.c.assimilator = get_assimilator(scheme.c)

scheme.c.logger('Assimilator')(scheme.c.assimilator.assimilate)(scheme.c)

In [ ]:
from NEDAS.assim_tools.updators import get_updator
scheme.c.updator = get_updator(scheme.c)
scheme.c.logger('Updator')(scheme.c.updator.update)(scheme.c)

In [ ]:
# compute correlation
fld_prior_ens = np.zeros((config.nens,)+grid.x.shape)
for m in range(config.nens):
    fld_prior_ens[m,...] = scheme.c.state.fields_prior[m,0][0,...]
fld_prior_mean = np.mean(fld_prior_ens, axis=0)

obs_prior_ens = np.array([scheme.c.obs.obs_prior[m,0][0] for m in range(config.nens)])
obs_prior_mean = np.mean(obs_prior_ens, axis=0)

cov = np.zeros(grid.x.shape)
fld_prior_var = np.zeros(grid.x.shape)
obs_prior_var = 0
for m in range(config.nens):
    cov += ((fld_prior_ens[m,...] - fld_prior_mean) * (obs_prior_ens[m] - obs_prior_mean))
    fld_prior_var += (fld_prior_ens[m,...] - fld_prior_mean)**2
    obs_prior_var += (obs_prior_ens[m] - obs_prior_mean)**2
cov /= config.nens-1
fld_prior_var /= config.nens-1
obs_prior_var /= config.nens-1

fld_prior_var[np.where(fld_prior_var==0)] = 1e-10

corr = cov / np.sqrt(fld_prior_var * obs_prior_var)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12,4), constrained_layout=True)
vmax = 30
state_i, state_j = 30, 30
state_x = grid.x[state_j, state_i]
state_y = grid.y[state_j, state_i]
state_prior = fld_prior_ens[:, state_j, state_i]

# truth field; obs; and state(i,j) location
i = times.index(scheme.c.time)
grid.plot_vectors(ax[0], truth_state[i], V=vmax, linecolor=[.7,.7,.7])                  
obs_rec_id = 0
obs_seq = scheme.c.obs.obs_seq[obs_rec_id]
obs_val = obs_seq['obs']
obs_x = obs_seq['x'] 
obs_y = obs_seq['y']
grid.plot_scatter(ax[0], obs_val, vmax=vmax, x=obs_x, y=obs_y, is_vector=True, linecolor='k', linewidth=1)
ax[0].plot(obs_x, obs_y, color='k', marker='o', markersize=3, zorder=10)
ax[0].plot(state_x, state_y, color='k', marker='+', markersize=10, zorder=10)
set_map_axis(ax[0])

# correlation map
cmap = getattr(cmocean.cm, 'balance') 
#grid.plot_field(ax[1], truth_state[i][0], -model.Vmax, model.Vmax, cmap)
#grid.plot_scatter(ax[1], obs_val[0], -model.Vmax, model.Vmax, 15, cmap, x=obs_x, y=obs_y)
grid.plot_field(ax[1], corr, vmin=-1, vmax=1, cmap=cmap)
#add_colorbar(fig, ax[1], cmap, -1, 1)
ax[1].plot(obs_x, obs_y, color='k', marker='o', markersize=3, zorder=10)
ax[1].plot(state_x, state_y, color='k', marker='+', markersize=10, zorder=10)
set_map_axis(ax[1])

# scatter plot of obs-state relation
ax[2].scatter(obs_prior_ens, state_prior)


## Running experiments

Running the entire scheme, cycling from time_start to time_end

A few things to play with

Since the step by step shown, only works with nproc=1

If you will try nproc>1, use scheme() directly


Case

configuration

run scheme

check results, visualizations and performance diagnostics

### Define cases

In [ ]:
# A free ensemble forecast without DA
casename = 'free_run'

config = Config(config_file='vort2d/config.yml', run_analysis=False, quiet=True)

In [4]:
casename = 'assim_global'

#config
config = Config(config_file='vort2d/config.yml', run_analysis=True, quiet=True)

In [5]:
scheme = get_scheme(config)

### Run the case and collect data

In [ ]:
# clear previous results
%rm -rf vort2d/work/cycle


In [ ]:
scheme()

In [14]:
truth_state = get_time_series(scheme.c, get_truth)
ens_state = get_time_series(scheme.c, get_model_ens)

### Check diagnostic plots

In [15]:
plot_ens_error_sprd(casename, scheme.c, truth_state, ens_state)

make_animation(casename, scheme.c)

# Display in notebook
display(HTML(f'<img src="vort2d/{casename}_diag_animation.gif">'))

In [16]:
# or, to view the plots in an interactive ui with a time slider
ui = interactive_animator(casename, scheme.c)
display(ui)